In [ ]:
import os
import sys
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from tqdm import tqdm
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

In [ ]:

# ==========================================
# 0. 路径配置（和你原有代码完全一致）
# ==========================================
current_dir = os.path.dirname(os.path.abspath(__file__))
root_dir = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.append(root_dir)

from model.model_minimind import MiniMindConfig, MiniMindForCausalLM
from dataset.lm_dataset import RLAIFDataset

device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 2
max_seq_len = 128
max_gen_len = 150

# Tokenizer
print("🚀 加载 Tokenizer...")
model_dir = os.path.join(root_dir, 'model')
tokenizer = AutoTokenizer.from_pretrained(model_dir)
tokenizer.pad_token_id = tokenizer.eos_token_id

In [ ]:

# ==========================================
# 1. GRPO 模型：只需要 Actor + Ref 模型（删除 Critic！）
# ==========================================
print("🧠 初始化 GRPO 模型 (Actor + Ref)")
ppo_config = MiniMindConfig(hidden_size=768, num_hidden_layers=8, num_attention_heads=8)

# 策略模型（训练目标）
actor = MiniMindForCausalLM(ppo_config).to(device)
# 参考模型（冻结，用于 KL 惩罚）
ref_model = MiniMindForCausalLM(ppo_config).to(device)

# 加载 SFT 权重
sft_model_path = os.path.join(root_dir, 'out', 'full_sft_768.pth')
if os.path.exists(sft_model_path):
    state_dict = torch.load(sft_model_path, map_location=device)
    actor.load_state_dict(state_dict, strict=False)
    ref_model.load_state_dict(state_dict, strict=False)
    print("✅ SFT 权重加载成功！")

# 模型模式
actor.train()
ref_model.eval().requires_grad_(False)

# ==========================================
# 2. 奖励模型（和你原有代码完全不变）
# ==========================================
import re
def rep_penalty(text, n=3, cap=0.5): # 3-gram
    toks = re.findall(r"\w+|[^\w\s]", text.lower())
    grams = [tuple(toks[i:i + n]) for i in range(len(toks) - n + 1)]
    return min(cap, (len(grams) - len(set(grams)))) * cap * 2 / len(grams) if grams else 0.0

class MockRewardModel:
    def __init__(self, device):
        self.device = device
    def get_reward(self, response_texts):
        rewards = []
        for response in response_texts:
            score = 0.5 if 20 <= len(response.strip()) <= 800 else -0.5
            if '</think>' in response:
                try:
                    think, ans = response.split('</think>', 1)
                    score += 1.0 if 20 <= len(think) <= 300 else -0.5
                    score -= rep_penalty(ans)
                except: score -= 0.5
            if "请" in response or "谢谢" in response: score += 0.5
            rewards.append(score)
        return torch.tensor(rewards, dtype=torch.bfloat16, device=device)

reward_model = MockRewardModel(device)
optimizer = AdamW(actor.parameters(), lr=1e-5)  # 只优化 Actor！

# ==========================================
# 🔥 GRPO 核心算法实现（唯一需要的损失函数）
# ==========================================
def grpo_loss(
    actor_logits, ref_logits, actions, 
    rewards, beta_kl=0.1, group_size=2
):
    """
    GRPO 核心损失：分组相对奖励 + KL 惩罚
    :param actor_logits: 当前策略输出 [B, T, V]
    :param ref_logits: 参考模型输出 [B, T, V]
    :param actions: 生成的回答 token [B, T]
    :param rewards: 奖励模型打分 [B]
    :param beta_kl: KL 惩罚系数
    :param group_size: 分组大小（和 batch_size 一致）
    """
    # 1. 计算策略对数概率
    actor_dist = torch.distributions.Categorical(logits=actor_logits)
    ref_dist = torch.distributions.Categorical(logits=ref_logits)
    
    log_pi = actor_dist.log_prob(actions)  # [B, T]
    log_ref = ref_dist.log_prob(actions)  # [B, T]

    # 2. 逐 token KL 散度
    kl = log_pi - log_ref  # 近似 KL 散度

    # 3. 分组相对奖励（GRPO 灵魂！）
    rewards = rewards.view(-1, 1)  # [B, 1]
    # 分组归一化：组内相对奖励（消除全局尺度影响）
    group_mean = rewards.view(-1, group_size).mean(dim=1, keepdim=True)
    group_std = rewards.view(-1, group_size).std(dim=1, keepdim=True) + 1e-8
    rel_rewards = (rewards - group_mean) / group_std  # 相对奖励

    # 4. 扩展到逐 token 维度
    rel_rewards = rel_rewards.expand_as(log_pi)  # [B, T]

    # 5. GRPO 损失（最大化目标 → 最小化负损失）
    loss = -(rel_rewards * (log_pi - beta_kl * kl)).mean()
    
    # 辅助损失（监控用）
    policy_loss = -(rel_rewards * log_pi).mean()
    kl_loss = kl.mean()
    
    return loss, policy_loss, kl_loss

# ==========================================
# 3. 数据加载（不变）
# ==========================================
dataset_path = os.path.join(root_dir, 'dataset', 'test_rlaif.jsonl')
train_ds = RLAIFDataset(jsonl_path=dataset_path, tokenizer=tokenizer, max_length=max_seq_len)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

# ==========================================
# 4. GRPO 训练循环（极简！无 GAE/无 Critic）
# ==========================================
# 超参数
EPOCHS = 1          # 外层遍历数据集次数
GRPO_EPOCHS = 4     # 内层更新轮次（PPO 同款 3~5）
BETA_KL = 0.1       # KL 惩罚系数

print("🚀 开始 GRPO 训练...")
for epoch in range(EPOCHS):
    pbar = tqdm(train_loader, desc=f"GRPO Epoch {epoch+1}/{EPOCHS}")
    
    for batch in pbar:
        # --------------------------
        # 阶段1：Rollout 生成回答（和 PPO 一致）
        # --------------------------
        prompt_ids = tokenizer(
            batch['prompt'], return_tensors="pt", padding=True, truncation=True
        ).input_ids.to(device)
        B, prompt_len = prompt_ids.shape
        
        actor.eval()
        with torch.no_grad():
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                # 生成回答
                output_ids = actor.generate(
                    prompt_ids, max_new_tokens=max_gen_len, 
                    do_sample=True, temperature=0.7,
                    pad_token_id=tokenizer.eos_token_id
                )
                generated_ids = output_ids[:, prompt_len:]  # 回答部分
                T = generated_ids.shape[1]
                
                # 计算奖励
                responses = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
                rewards = reward_model.get_reward(responses)
                
                # 切片 logits（修复版）
                start = prompt_len
                end = start + T
                actor_logits_old = actor(output_ids).logits[:, start:end, :]
                ref_logits_old = ref_model(output_ids).logits[:, start:end, :]

        # --------------------------
        # 阶段2：GRPO 多轮更新（核心）
        # --------------------------
        actor.train()
        train_input_ids = torch.cat([prompt_ids, generated_ids], dim=1)
        
        for _ in range(GRPO_EPOCHS):
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                # 前向传播
                new_logits = actor(train_input_ids).logits[:, start:end, :]
                # 计算 GRPO 损失
                loss, p_loss, kl_loss = grpo_loss(
                    actor_logits=new_logits,
                    ref_logits=ref_logits_old,
                    actions=generated_ids,
                    rewards=rewards,
                    beta_kl=BETA_KL,
                    group_size=batch_size
                )

            # 反向传播
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(actor.parameters(), 1.0)
            optimizer.step()

        # 日志
        pbar.set_postfix({
            "Reward": f"{rewards.mean().item():.2f}",
            "Loss": f"{loss.item():.3f}",
            "KL": f"{kl_loss.item():.3f}"
        })

# ==========================================
# 5. 保存模型（只保存 Actor）
# ==========================================
save_path = os.path.join(root_dir, 'out', 'grpo_final.pth')
torch.save(actor.state_dict(), save_path)
print(f"\n🎉 GRPO 训练完成！模型保存至: {save_path}")